In [24]:
# === IMPORT CONFIGURATION ===
import os
from pathlib import Path

# Import settings from config
from config import (
    BASE_DIR, DATA_DIR, LOGS_DIR, TRANSCRIPTS_DIR, RAW_DIR,
    OUTPUT_DIR, CLEANED_TRANSCRIPTS, TRANSCRIPT_METADATA
)

# Check if we're in the right place and if the directories exist
print(f"Current working directory: {os.getcwd()}")
print(f"Checking if transcripts directory exists: {os.path.exists(TRANSCRIPTS_DIR)}")

# Import all the packages we need
import sys
import pandas as pd
import numpy as np
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import datetime

# Get NLTK data we need for sentiment analysis
nltk.download('vader_lexicon')
nltk.download('punkt')

print("Starting transcript data cleaning process...")

# Basic text cleaning to fix HTML and formatting issues
def clean_text(text):
    """Cleans text by removing HTML tags and fixing spacing"""
    if not isinstance(text, str) or pd.isna(text):
        return ""

    # Fix HTML special characters
    text = re.sub(r'&nbsp;', ' ', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)

    # Get rid of HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Fix extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Custom sentiment analyzer that works better for financial texts
def perform_sentiment_analysis(text):
    """Analyzes sentiment but tweaked to work better for earnings calls"""
    if not isinstance(text, str) or pd.isna(text):
        return 0.5, 0, 0  # Default to neutral if no text

    # Remove legal disclaimers that can skew sentiment
    disclaimer_patterns = [
        r"forward-looking statements.*?risks and uncertainties",
        r"safe harbor.*?forward-looking",
        r"please note.*?forward-looking"
    ]

    cleaned_text = text
    for pattern in disclaimer_patterns:
        cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.IGNORECASE | re.DOTALL)

    # Replace financial terms that VADER might misinterpret
    financial_terms = ["liability", "debt", "expense", "tax", "margin", "guidance"]
    for term in financial_terms:
        cleaned_text = re.sub(r'\b' + term + r'\b', "item", cleaned_text, flags=re.IGNORECASE)

    # Run the sentiment analyzer on each sentence
    analyzer = SentimentIntensityAnalyzer()
    sentences = nltk.sent_tokenize(cleaned_text)

    scores = []
    positive_count = 0
    negative_count = 0

    for sentence in sentences:
        if len(sentence.strip()) < 10:
            continue

        score = analyzer.polarity_scores(sentence)
        scores.append(score['compound'])

        if score['compound'] >= 0.05:
            positive_count += 1
        elif score['compound'] <= -0.05:
            negative_count += 1

    # Make the sentiment scores more useful by spreading them out
    if scores:
        raw_score = sum(scores) / len(scores)
        # Stretch the scores out to use more of the 0-1 range
        adjusted_score = 0.5 + (raw_score * 0.4)  # Makes scores range from ~0.1-0.9
        return adjusted_score, positive_count, negative_count

    return 0.5, 0, 0

# Pull out forward-looking statements about future performance
def extract_outlook(text):
    """Finds parts of the transcript that talk about future outlook"""
    if not isinstance(text, str) or pd.isna(text):
        return ""

    # Words that usually signal talking about the future
    outlook_keywords = [
        "outlook", "guidance", "forward-looking", "future", "expect", 
        "anticipate", "forecast", "projected", "next quarter", "next year",
        "looking ahead", "going forward"
    ]

    sentences = re.split(r'\.|\?|\!', text)
    outlook_sentences = []

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if any(keyword in sentence.lower() for keyword in outlook_keywords):
            outlook_sentences.append(sentence + ".")

    # Just return the first few to keep it manageable
    return " ".join(outlook_sentences[:5])

# Find mentions of products in the transcript
def extract_product_mentions(text):
    """Finds sentences talking about products or services"""
    if not isinstance(text, str) or pd.isna(text):
        return ""

    # Find sentences about products and revenue
    revenue_pattern = r'([^.!?]*?revenue[^.!?]*?)[.!?]'
    product_pattern = r'([^.!?]*?product[^.!?]*?)[.!?]'
    service_pattern = r'([^.!?]*?service[^.!?]*?)[.!?]'

    matches = []

    for pattern in [revenue_pattern, product_pattern, service_pattern]:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            matches.append(match.group(1).strip())

    # Keep it brief - just the first couple matches
    return " ".join(matches[:2])

# Try to extract revenue figures with high accuracy
def extract_revenue(text):
    """Finds revenue numbers in the transcript with specific patterns"""
    if not isinstance(text, str) or pd.isna(text):
        return None

    # Skip the header which might have irrelevant numbers
    if len(text) > 200:
        text_to_search = text[200:]
    else:
        text_to_search = text

    # Look for very specific revenue mentions
    strict_patterns = [
        r'revenue was \$?(\d+\.?\d*)\s*(billion|b)',
        r'revenue of \$?(\d+\.?\d*)\s*(billion|b)',
        r'revenue: \$?(\d+\.?\d*)\s*(billion|b)',
        r'revenue was \$?(\d+\.?\d*)\s*(million|m)',
        r'revenue of \$?(\d+\.?\d*)\s*(million|m)',
        r'revenue: \$?(\d+\.?\d*)\s*(million|m)',
        r'total revenue of \$?(\d+\.?\d*)\s*(billion|b)',
        r'total revenue: \$?(\d+\.?\d*)\s*(billion|b)',
        r'reported revenue of \$?(\d+\.?\d*)\s*(billion|b)',
    ]

    for pattern in strict_patterns:
        match = re.search(pattern, text_to_search.lower(), re.IGNORECASE)
        if match:
            try:
                value = float(match.group(1))
                unit = match.group(2).lower()

                if 'billion' in unit or 'b' == unit:
                    return value
                elif 'million' in unit or 'm' == unit:
                    return value / 1000  # Convert to billions
            except:
                pass

    # If we didn't find anything exact, try a more flexible approach
    sentences = re.split(r'\.|\?|\!', text_to_search)
    for sentence in sentences:
        if 'revenue' in sentence.lower() and '$' in sentence:
            # Look for dollar amounts in sentences about revenue
            dollar_match = re.search(r'\$\s*(\d+\.?\d*)', sentence)
            if dollar_match:
                try:
                    value = float(dollar_match.group(1))
                    # Guess if it's billions or millions based on size
                    if value < 100:  # Probably billions
                        return value
                    else:  # Probably millions
                        return value / 1000
                except:
                    pass

    return None

# Try to extract net income figures accurately
def extract_net_income(text):
    """Finds net income numbers in the transcript"""
    if not isinstance(text, str) or pd.isna(text):
        return None

    # Skip the header which might have irrelevant numbers
    if len(text) > 200:
        text_to_search = text[200:]
    else:
        text_to_search = text

    # Look for specific mentions of net income
    strict_patterns = [
        r'net income was \$?(\d+\.?\d*)\s*(billion|b)',
        r'net income of \$?(\d+\.?\d*)\s*(billion|b)',
        r'net income: \$?(\d+\.?\d*)\s*(billion|b)',
        r'net income was \$?(\d+\.?\d*)\s*(million|m)',
        r'net income of \$?(\d+\.?\d*)\s*(million|m)',
        r'net income: \$?(\d+\.?\d*)\s*(million|m)',
        r'reported net income of \$?(\d+\.?\d*)\s*(billion|b)',
        r'reported net income: \$?(\d+\.?\d*)\s*(billion|b)',
    ]

    for pattern in strict_patterns:
        match = re.search(pattern, text_to_search.lower(), re.IGNORECASE)
        if match:
            try:
                value = float(match.group(1))
                unit = match.group(2).lower()

                if 'billion' in unit or 'b' == unit:
                    return value
                elif 'million' in unit or 'm' == unit:
                    return value / 1000  # Convert to billions
            except:
                pass

    # If we didn't find anything exact, try a more flexible approach
    sentences = re.split(r'\.|\?|\!', text_to_search)
    for sentence in sentences:
        if 'net income' in sentence.lower() and '$' in sentence:
            # Look for dollar amounts in sentences about net income
            dollar_match = re.search(r'\$\s*(\d+\.?\d*)', sentence)
            if dollar_match:
                try:
                    value = float(dollar_match.group(1))
                    # Guess if it's billions or millions based on size
                    if value < 100:  # Probably billions
                        return value
                    else:  # Probably millions
                        return value / 1000
                except:
                    pass

    return None

# New method to parse quarter from filename or text
def extract_quarter(filename, text):
    """Tries to extract the quarter information from filename or text content"""
    # First try from filename (TICKER_DATE_QUARTER_source.txt)
    parts = filename.split('_')
    if len(parts) >= 3:
        quarter_part = parts[2]
        if 'Q' in quarter_part:
            return quarter_part

    # Try to find quarter in text
    quarter_pattern = r'Q[1-4]\s+\d{4}'
    match = re.search(quarter_pattern, text)
    if match:
        return match.group(0)

    # If we can't find it, try to infer from date
    if len(parts) >= 2:
        try:
            date = pd.to_datetime(parts[1])
            month = date.month
            year = date.year
            if month <= 3:
                return f"Q1 {year}"
            elif month <= 6:
                return f"Q2 {year}"
            elif month <= 9:
                return f"Q3 {year}"
            else:
                return f"Q4 {year}"
        except:
            pass

    return "Unknown"

# Main data processing starts here
# Load the data directly from transcript files instead of metadata
print("Loading transcript files directly from directory...")
transcript_data = []

# Get all files in the transcripts directory
transcript_files = list(Path(TRANSCRIPTS_DIR).glob('*.txt'))

# FILTER OUT SOURCE FILES - This is the fix for duplicates
transcript_files = [f for f in transcript_files if '_source.txt' not in str(f)]

print(f"Found {len(transcript_files)} transcript files in directory")



# Process each transcript file
for file_path in transcript_files:
    try:
        # Extract information from filename (assuming format TICKER_DATE_etc.txt)
        filename = file_path.name
        parts = filename.split('_')

        if len(parts) >= 2:
            ticker = parts[0]
            date_str = parts[1]

            # Read the file content
            with open(file_path, 'r', encoding='utf-8') as file:
                text = file.read()

            # Try to parse the date
            try:
                filing_date = pd.to_datetime(date_str).strftime('%Y-%m-%d')
            except:
                # Try to find date in text if filename doesn't work
                date_pattern = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},\s+\d{4}'
                date_match = re.search(date_pattern, text)
                if date_match:
                    try:
                        filing_date = pd.to_datetime(date_match.group(0)).strftime('%Y-%m-%d')
                    except:
                        filing_date = date_str  # Keep as is if can't parse
                else:
                    filing_date = date_str

            # Extract quarter information
            quarter = extract_quarter(filename, text)

            # Extract company name from text
            company_name = ticker  # Default to ticker
            if len(text) > 0:
                first_line = text.split('\n')[0]
                if len(first_line) > 0 and ticker in first_line:
                    company_parts = first_line.split(ticker)
                    if len(company_parts) > 0 and len(company_parts[0].strip()) > 0:
                        company_name = company_parts[0].strip()

            # Create a record for this transcript
            transcript_data.append({
                'ticker': ticker,
                'company_name': company_name,
                'filing_date': filing_date,
                'quarter': quarter,
                'transcript_file': str(file_path),
                'transcript_length': len(text),
                'transcript_text': text,
                'source': 'file',
                'source_url': ''
            })

            print(f"Loaded: {filename}")

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Create DataFrame from the collected data
if transcript_data:
    transcript_df = pd.DataFrame(transcript_data)
    print(f"Loaded {len(transcript_df)} transcripts with text content")
else:
    print("No transcript files found! Check the directory path.")
    transcript_df = pd.DataFrame(columns=['ticker', 'company_name', 'filing_date', 'quarter', 
                                         'transcript_file', 'transcript_length', 'transcript_text'])

# Clean up the text to remove HTML and fix formatting
print("Cleaning transcript text...")
transcript_df['cleaned_text'] = transcript_df['transcript_text'].apply(clean_text)

# Run our sentiment analyzer on each transcript
print("Performing sentiment analysis...")
sentiment_results = transcript_df['cleaned_text'].apply(perform_sentiment_analysis)

transcript_df['sentiment_score'] = [result[0] for result in sentiment_results]
transcript_df['positive_count'] = [result[1] for result in sentiment_results]
transcript_df['negative_count'] = [result[2] for result in sentiment_results]

# Pull out key information from each transcript
print("Extracting key information from transcripts...")
transcript_df['outlook'] = transcript_df['transcript_text'].apply(extract_outlook)
transcript_df['product_mentions'] = transcript_df['transcript_text'].apply(extract_product_mentions)

# Clean the extracted information
print("Cleaning extracted information...")
transcript_df['outlook_cleaned'] = transcript_df['outlook'].apply(clean_text)
transcript_df['product_mentions_cleaned'] = transcript_df['product_mentions'].apply(clean_text)

# Extract financial numbers from the text
print("Extracting financial metrics...")

# Show some examples of revenue mentions to help fine-tune the extraction
print("\nExamining revenue mentions to help improve extraction:")
for idx, row in transcript_df.sample(min(5, len(transcript_df))).iterrows():
    revenue_mention = extract_product_mentions(row['transcript_text'])
    if revenue_mention:
        print(f"\nTicker: {row['ticker']}")
        print(f"Revenue mention: {revenue_mention[:150]}...")

print("\nCreating revenue_billions column with enhanced extraction...")
transcript_df['revenue_billions'] = transcript_df['transcript_text'].apply(extract_revenue)
revenue_count = transcript_df['revenue_billions'].notna().sum()
print(f"Created revenue_billions column with {revenue_count} values out of {len(transcript_df)} rows")

# Show some examples of earnings mentions to help fine-tune the extraction
print("\nExamining earnings mentions to help improve extraction:")
for idx, row in transcript_df.sample(min(5, len(transcript_df))).iterrows():
    earnings_text = row['transcript_text'][:150]
    if 'earnings' in earnings_text.lower():
        print(f"\nTicker: {row['ticker']}")
        print(f"Earnings mention: {earnings_text}...")

print("\nCreating net_income_billions column with enhanced extraction...")
transcript_df['net_income_billions'] = transcript_df['transcript_text'].apply(extract_net_income)
income_count = transcript_df['net_income_billions'].notna().sum()
print(f"Created net_income_billions column with {income_count} values out of {len(transcript_df)} rows")

# Standardize our financial columns
print("Standardizing financial values...")
transcript_df['revenue_billions_std'] = transcript_df['revenue_billions']
transcript_df['net_income_billions_std'] = transcript_df['net_income_billions']

# Keep NaN values as NaN, not 0
transcript_df['revenue_billions'] = transcript_df['revenue_billions'].fillna(np.nan)
transcript_df['net_income_billions'] = transcript_df['net_income_billions'].fillna(np.nan)
transcript_df['revenue_billions_std'] = transcript_df['revenue_billions_std'].fillna(np.nan)
transcript_df['net_income_billions_std'] = transcript_df['net_income_billions_std'].fillna(np.nan)

# Make sure dates are formatted consistently
print("Formatting dates...")
if 'filing_date' in transcript_df.columns and transcript_df['filing_date'].dtype == 'object':
    try:
        transcript_df['filing_date'] = pd.to_datetime(transcript_df['filing_date']).dt.strftime('%Y-%m-%d')
    except:
        pass

# Add file path column for backward compatibility
transcript_df['file_path'] = transcript_df['transcript_file']

# Define columns to keep
print("Preparing final dataset...")
columns_to_keep = [
    'ticker', 'company_name', 'filing_date', 'quarter', 
    'transcript_file', 'transcript_length', 'source', 'source_url',
    'file_path', 'sentiment_score', 'positive_count', 'negative_count',
    'outlook_cleaned', 'product_mentions_cleaned',
    'revenue_billions', 'net_income_billions',
    'revenue_billions_std', 'net_income_billions_std'
]

# Only keep columns that actually exist in our dataframe
columns_to_keep = [col for col in columns_to_keep if col in transcript_df.columns]
cleaned_df = transcript_df[columns_to_keep]

# Save our cleaned data
print("Saving cleaned dataset...")
cleaned_df.to_csv(CLEANED_TRANSCRIPTS, index=False)
print(f"Saved cleaned data to: {CLEANED_TRANSCRIPTS}")

# Show some stats about our dataset
print("\nData Cleaning Complete!")
print(f"Total transcripts processed: {len(cleaned_df)}")
print(f"Companies represented: {cleaned_df['ticker'].nunique()}")

if 'filing_date' in cleaned_df.columns:
    try:
        min_date = pd.to_datetime(cleaned_df['filing_date']).min().strftime('%Y-%m-%d')
        max_date = pd.to_datetime(cleaned_df['filing_date']).max().strftime('%Y-%m-%d')
        print(f"Date range: {min_date} to {max_date}")
    except:
        pass

print(f"Sentiment score range: {cleaned_df['sentiment_score'].min():.2f} to {cleaned_df['sentiment_score'].max():.2f}")
print(f"Average sentiment score: {cleaned_df['sentiment_score'].mean():.2f}")

revenue_pct = (cleaned_df['revenue_billions'].notna().sum() / len(cleaned_df)) * 100
income_pct = (cleaned_df['net_income_billions'].notna().sum() / len(cleaned_df)) * 100

print(f"Transcripts with revenue data: {cleaned_df['revenue_billions'].notna().sum()} ({revenue_pct:.1f}%)")
print(f"Transcripts with income data: {cleaned_df['net_income_billions'].notna().sum()} ({income_pct:.1f}%)")

print("\nSample of cleaned data:")
print(cleaned_df.sample(min(5, len(cleaned_df)))[['ticker', 'filing_date', 'sentiment_score', 'revenue_billions_std', 'net_income_billions_std']])

print("\nCleaning process complete! Files saved in data/ directory.")


Current working directory: C:\Users\luke3\Documents\GitHub\Earnings Call Analyzer\notebooks
Checking if transcripts directory exists: True
Starting transcript data cleaning process...
Loading transcript files directly from directory...
Found 140 transcript files in directory
Loaded: AAPL_2023-08-02_Q3 2023_seeking_alpha.txt
Loaded: AAPL_2023-10-31_Q4 2023_seeking_alpha.txt
Loaded: AAPL_2024-01-29_Q1 2024_seeking_alpha.txt
Loaded: AAPL_2024-04-28_Q2 2024_seeking_alpha.txt
Loaded: AAPL_2024-07-27_Q3 2024_seeking_alpha.txt
Loaded: AAPL_2024-10-25_Q4 2024_seeking_alpha.txt
Loaded: AAPL_2025-01-23_Q1 2025_seeking_alpha.txt
Loaded: AMZN_2023-08-02_Q3 2023_seeking_alpha.txt
Loaded: AMZN_2023-10-31_Q4 2023_seeking_alpha.txt
Loaded: AMZN_2024-01-29_Q1 2024_seeking_alpha.txt
Loaded: AMZN_2024-04-28_Q2 2024_seeking_alpha.txt
Loaded: AMZN_2024-07-27_Q3 2024_seeking_alpha.txt
Loaded: BAC_2023-08-02_Q3 2023_seeking_alpha.txt
Loaded: BAC_2023-10-31_Q4 2023_seeking_alpha.txt
Loaded: BAC_2024-01-29_Q1 

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\luke3\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\luke3\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Extracting key information from transcripts...
Cleaning extracted information...
Extracting financial metrics...

Examining revenue mentions to help improve extraction:

Ticker: BAC
Revenue mention: 01 Â |Â Revenue of $25...

Ticker: AAPL
Revenue mention: 07 Â |Â Revenue of $81 Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, in...

Ticker: GS
Revenue mention: 21 Â |Â Revenue of $11...

Ticker: AAPL
Revenue mention: 05 Â |Â Revenue of $124 Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, i...

Ticker: AAPL
Revenue mention: 02 Â |Â Revenue of $90 Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, in...

Creating revenue_billions column with enhanced extraction...
Created revenue_billions column with 138 values out of 140 rows

Examining earnings mentions to help im